## Data Ingestion
So far, our RAG pipeline loads data and builds the search index at startup. With minsearch, this is fine - our FAQ dataset is small, so indexing takes less than a second. The entire pipeline runs in one process.

This breaks down as the dataset grows. Fetching data takes time - calling APIs, parsing files, cleaning text. With millions of documents, the startup becomes slow. You don't want to wait minutes every time your service restarts.

Minsearch is in-memory. It's a bunch of Python dictionaries bound to the process where it's running. When you stop the process, the data disappears, so you re-index every time you restart. That's wasteful if indexing is slow or the data takes time to prepare.

So we separate ingestion from querying. One process writes the data to a persistent search index. Another process reads from it. The two run independently and only share the index between them.

The index survives restarts, so we ingest once and query as often as we like. This is the ingestion step from data engineering. We move data from its source into a target system the application can use.

You can use any persistent search backend for this, such as Elasticsearch, OpenSearch, or Qdrant. In this module, we use sqlitesearch, a lightweight search library backed by SQLite FTS5. It has the same API as minsearch, so it's a drop-in replacement that happens to be persistent.

It ships with Python, so you don't add any dependency, and it has FTS5 (full text search) built in. If you have Python, you already have a full text search engine. Using FTS5 directly is a bit awkward, so sqlitesearch is a convenient wrapper around it.


### Ingestion notebook
First, load the data using the function from ingest.py and filter to just LLM Zoomcamp documents

In [ ]:
from ingest import load_faq_data

documents = load_faq_data()
print(f"Loaded {len(documents)} documents")

Loaded 1349 documents


In [ ]:
docs_llm = [doc for doc in documents if doc["course"] == "llm-zoomcamp"]
print(f"LLM Zoomcamp: {len(docs_llm)} documents")

LLM Zoomcamp: 84 documents


Now create a sqlitesearch index and add documents one by one with a small delay (to simulate slow ingestion):

In [ ]:
import time
from sqlitesearch import TextSearchIndex

index = TextSearchIndex(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"],
    db_path="faq.db"
)

for doc in docs_llm:
    index.add(doc)
    print(f"""Added: {doc["question"][:60]}...""")
    time.sleep(0.5)

index.close()
print("Done. Index saved to faq.db")

Added: I just discovered the course. Can I still join?...
Added: Course: I have registered for the LLM Zoomcamp. When can I e...
Added: What is the video/zoom link to the stream for the “Office Ho...
Added: How should I start the course and follow the weekly workflow...
Added: Leaderboard: I am not on the leaderboard / how do I know whi...
Added: Certificate: Can I follow the course in a self-paced mode an...
Added: I missed the first homework - can I still get a certificate?...
Added: Homework: Why does the content keep changing?...
Added: When will the course be offered next?...
Added: Are there any lectures/videos? Where are they?...
Added: Where can I track the LLM Zoomcamp syllabus, deadlines, home...
Added: Are there live sessions or office hours for each module?...
Added: Can I use Bluesky for learning in public credits?...
Added: Where is the LLM Zoomcamp Telegram channel?...
Added: Why are we not using Langchain in the course?...
Added: OpenAI: Error when running OpenAI respon

When it's done, there's a faq.db file on disk with the entire index. This file persists across restarts.

### RAG with sqlitesearch
We use the RAGBase class from rag_helper.py with this sqlitesearch index. Because our RAG is modular, we just swap the search index - the rest of the code stays the same:

In [ ]:
from rag_helper import RAGBase
from openai import OpenAI

openai_client = OpenAI()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)

This code skips both the fit call and the data loading. The index is already populated by the ingestion process, so we just connect to the database file.

In [ ]:
answer = assistant.rag("Can I still join the course after it started?")
print(answer)

Yes, you can still join the course after it started. If you want to receive a certificate, make sure to submit your project while submissions are still being accepted.


The answer should be similar to what we got with minsearch. But now the data comes from a persistent index - no fetching, no processing, no indexing at startup. And we didn't have to rewrite any of the RAG logic - just swapped the index.

The modular design splits the work cleanly:

- ingest.py handles data loading and indexing
- rag_helper.py handles the RAG pipeline
- the notebooks wire them together

This works because sqlitesearch follows the same API as minsearch - both have a search method that takes a query, boost_dict, filter_dict, and num_results. If the API were different, we'd need to subclass RAGBase and override the search method to adapt to the new backend.


In [ ]:
index.count()

84

### Cleaning up
When you're done, close the database connection, or just let Python clean it up when the notebook kernel shuts down.

In [ ]:
index.close()